# Softmax (multiclass)

_Machine Learning_

**Turn many scores into probabilities that add up to 1.**

Sigmoid handles two classes. Softmax handles many classes.

     Each class gets its own score. Softmax turns the scores into probabilities.

---

This notebook builds softmax step by step. Run each cell, read the note above it, and watch how a handful of raw scores become a clean set of probabilities. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_iris

data = load_iris(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — PyTorch

We train a linear model that produces one score per class, then turn those scores into probabilities with softmax. We build it in three steps: (1) set up the model, loss, and data, (2) run the training loop, (3) read off the probabilities and accuracy.

### Step 1 — Set up the model, loss, and data

A `nn.Linear(4, K)` layer maps each 4-feature input to `K = 3` raw scores (one per class). `CrossEntropyLoss` conveniently bundles softmax and the cross-entropy penalty into a single step, so we feed it the raw scores directly.

We generate 300 random inputs and build labels in `{0, 1, 2}` from two simple thresholds, giving us a three-class problem to learn.

In [ ]:
import torch
import torch.nn as nn

K = 3                                # number of classes
model = nn.Linear(4, K)              # one score per class
loss_fn = nn.CrossEntropyLoss()      # softmax + cross-entropy in one
opt = torch.optim.SGD(model.parameters(), lr=0.1)

X = torch.randn(300, 4)

# Build labels in {0, 1, 2} from two thresholds on the features.
class_a = (X[:, 0] + 2 * X[:, 1] > 0).long()
class_b = (X[:, 2] > 1).long()
y = class_a + class_b

### Step 2 — Train the model

Each epoch we clear old gradients, compute the raw scores (`logits`), measure the cross-entropy loss, back-propagate to get gradients, and let the optimizer nudge the weights downhill. Over 150 epochs the scores gradually shift so the correct class gets the highest score.

In [ ]:
for epoch in range(150):
    opt.zero_grad()
    logits = model(X)               # raw scores, shape (300, K)
    loss = loss_fn(logits, y)
    loss.backward()
    opt.step()

### Step 3 — Turn scores into probabilities

`torch.softmax` converts each row of raw scores into probabilities that sum to 1. The predicted class is simply the one with the highest probability (`argmax`). We print the accuracy and the three probabilities softmax assigns to the first example.

In [ ]:
probs = torch.softmax(model(X), dim=1)   # each row sums to 1
preds = probs.argmax(dim=1)

accuracy = (preds == y).float().mean().item()
print("accuracy:", accuracy)
print("first row probs:", probs[0].detach().tolist())

## Visualize the data & results

_Shown one real iris flower's four measurements, what probability does softmax give each of the three species?_

### Step 1 — Fit softmax regression on real irises

We load the 150 real iris flowers (four measurements each, three species) and fit a `LogisticRegression`, which for multiple classes is exactly softmax regression. The fitted model can output a probability for each species.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

# 150 real flowers, 4 measurements, 3 species
iris = load_iris()
clf = LogisticRegression(max_iter=5000).fit(iris.data, iris.target)

### Step 2 — Plot the probabilities for one flower

We pick a single real flower and ask the model for its softmax probabilities across the three species. The bar chart shows how the model splits its confidence — the bars sum to 1, with the tallest being the predicted species.

In [ ]:
# softmax probabilities for one real flower
probs = clf.predict_proba(iris.data[60:61])[0]

plt.bar(list(iris.target_names), probs,
        color=["#ffb454", "#4ea1ff", "#c89bff"])
plt.ylabel("probability")
plt.title("Softmax species probabilities for one iris")
plt.show()